In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0]  # from notebooks/ up one level
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('C:/AMDARI PROJECTS/PREDICTIVE HEALTHCARE INTELLIENCE')

In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_loader import load_all_data
from src.utils import standardize_columns, parse_dates

sns.set(style="whitegrid")

data = load_all_data()
for name in data:
    data[name] = standardize_columns(data[name])

patients = data["patients"].copy()
admissions = data["admissions"].copy()
readmissions = data["readmissions"].copy()

In [7]:
# parse dates

admissions = parse_dates(admissions, ["admission_date", "discharge_date"])
readmissions = parse_dates(readmissions, ["original_discharge_date", "readmission_date"])
patients = parse_dates(patients, ["date_of_birth", "registered_date"])

In [8]:
readmissions.columns.tolist()

['readmission_id',
 'original_admission_id',
 'patient_id',
 'original_discharge_date',
 'readmission_date',
 'days_to_readmission',
 'readmission_type',
 'readmission_reason',
 'same_diagnosis',
 'planned_readmission',
 'avoided_if_discharged_better']

In [9]:
readmissions.head()


,readmission_id,original_admission_id,patient_id,original_discharge_date,readmission_date,days_to_readmission,readmission_type,readmission_reason,same_diagnosis,planned_readmission,avoided_if_discharged_better
0,RA-000001,ADM-0000005,PAT-004116,2020-07-17,2020-07-31,14,Urgent,New condition,1,0,0
1,RA-000002,ADM-0000006,PAT-002464,2022-09-03,2022-09-24,21,Emergency,Infection,0,1,0
2,RA-000003,ADM-0000019,PAT-001571,2022-01-15,2022-01-26,11,Emergency,New condition,0,0,1
3,RA-000004,ADM-0000020,PAT-004863,2021-03-07,2021-03-27,20,Emergency,Infection,1,0,1
4,RA-000005,ADM-0000032,PAT-003528,2020-10-12,2020-10-26,14,Emergency,Complication,1,1,1


In [10]:
# sanity checks

patients[["patient_id", "age"]].head()

admissions[["admission_id", "patient_id", "admission_date", "discharge_date"]].head()

readmissions[[
    "readmission_id",
    "original_admission_id",
    "patient_id",
    "original_discharge_date",
    "readmission_date"
]].head()

,readmission_id,original_admission_id,patient_id,original_discharge_date,readmission_date
0,RA-000001,ADM-0000005,PAT-004116,2020-07-17,2020-07-31
1,RA-000002,ADM-0000006,PAT-002464,2022-09-03,2022-09-24
2,RA-000003,ADM-0000019,PAT-001571,2022-01-15,2022-01-26
3,RA-000004,ADM-0000020,PAT-004863,2021-03-07,2021-03-27
4,RA-000005,ADM-0000032,PAT-003528,2020-10-12,2020-10-26


In [11]:
data = load_all_data()
for name in data:
    data[name] = standardize_columns(data[name])

patients = data["patients"].copy()
admissions = data["admissions"].copy()
readmissions = data["readmissions"].copy()

admissions = parse_dates(admissions, ["admission_date", "discharge_date"])
readmissions = parse_dates(readmissions, ["original_discharge_date", "readmission_date"])
patients = parse_dates(patients, ["date_of_birth", "registered_date"])

# Rename for consistent joining
readmissions = readmissions.rename(columns={"original_admission_id": "admission_id"})

In [12]:
# Building the admissions base table

base = admissions.merge(
    patients,
    on="patient_id",
    how="left",
    suffixes=("", "_patient")
)

# Attach readmission details (one-to-one on admission_id)
base = base.merge(
    readmissions[[
        "admission_id",
        "readmission_id",
        "readmission_date",
        "days_to_readmission",
        "planned_readmission",
        "avoided_if_discharged_better",
        "readmission_type",
        "readmission_reason",
        "same_diagnosis"
    ]],
    on="admission_id",
    how="left"
)

base.head()

,admission_id,patient_id,admission_date,discharge_date,length_of_stay_days,admission_type,admission_source,hospital,ward,primary_diagnosis_icd,...,social_support_score,registered_date,readmission_id,readmission_date,days_to_readmission,planned_readmission,avoided_if_discharged_better,readmission_type,readmission_reason_y,same_diagnosis
0,ADM-0000001,PAT-004458,2022-10-03,2022-10-12,9,Emergency,Physician Referral,MHN Cambridge,Geriatrics,N18.3,...,8,2018-03-28,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,ADM-0000002,PAT-001044,2023-06-24,2023-06-30,6,Urgent,Transfer,MHN Quincy,Cardiology,I10,...,9,2020-07-22,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
2,ADM-0000003,PAT-000197,2024-07-16,2024-07-19,3,Observation,Transfer,MHN Fenway,ICU,I21.9,...,10,2020-12-03,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
3,ADM-0000004,PAT-001374,2020-06-23,2020-06-29,6,Emergency,Direct Admission,MHN Cambridge,Orthopedics,I63.9,...,9,2021-11-28,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4,ADM-0000005,PAT-004116,2020-07-13,2020-07-17,4,Elective,Physician Referral,MHN Fenway,Respiratory,S72.001,...,10,2017-09-04,RA-000001,2020-07-31,14.0,0.0,0.0,Urgent,New condition,1.0


In [13]:
# Binary: any 30-day readmission based on admissions table (if it has that flag)
target_col = "readmitted_within_30d"

# Unplanned readmission indicator using readmissions table
base["unplanned_readmission_30d"] = (
    (base["readmission_date"].notna())
    & (base["planned_readmission"] == 0)
)

In [14]:
# Base binary target from admissions table
target_col = "readmitted_within_30d"

base[target_col].value_counts(dropna=False), base[target_col].mean()

(readmitted_within_30d
 0    6687
 1    1813
 Name: count, dtype: int64,
 np.float64(0.21329411764705883))

In [15]:
base["unplanned_readmission_30d"] = (
    base["readmission_date"].notna()
    & (base["planned_readmission"] == 0)
)

base["unplanned_readmission_30d"].value_counts(dropna=False), base["unplanned_readmission_30d"].mean()

(unplanned_readmission_30d
 False    6912
 True     1588
 Name: count, dtype: int64,
 np.float64(0.1868235294117647))

In [16]:
# Age bands vs readmission

base["age_band"] = pd.cut(
    base["age"],
    bins=[0, 49, 64, 74, 120],
    labels=["<50", "50-64", "65-74", "75+"]
)

age_readmit = (
    base.groupby("age_band")[target_col]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"mean": "readmission_rate"})
)

age_readmit

C:\Users\FOCO KONCEPT\AppData\Local\Temp\ipykernel_25940\822004904.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  base.groupby("age_band")[target_col]


,age_band,count,readmission_rate
0,<50,1671,0.164572
1,50-64,3200,0.218750
2,65-74,1688,0.232227
3,75+,1941,0.229778


In [17]:
# unplanned_readmission_30d

age_unplanned = (
    base.groupby("age_band")["unplanned_readmission_30d"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"mean": "unplanned_readmission_rate"})
)

age_unplanned

C:\Users\FOCO KONCEPT\AppData\Local\Temp\ipykernel_25940\3455773023.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  base.groupby("age_band")["unplanned_readmission_30d"]


,age_band,count,unplanned_readmission_rate
0,<50,1671,0.146020
1,50-64,3200,0.190000
2,65-74,1688,0.203791
3,75+,1941,0.201958


In [18]:
# LOS bands vs readmission

base["los_band"] = pd.cut(
    base["length_of_stay_days"],
    bins=[-1, 2, 5, 10, 20, 100],
    labels=["0-2", "3-5", "6-10", "11-20", "21+"]
)

los_readmit = (
    base.groupby("los_band")[target_col]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"mean": "readmission_rate"})
)

los_readmit

C:\Users\FOCO KONCEPT\AppData\Local\Temp\ipykernel_25940\782471783.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  base.groupby("los_band")[target_col]


,los_band,count,readmission_rate
0,0-2,1946,0.212744
1,3-5,3200,0.226875
2,6-10,2267,0.200265
3,11-20,915,0.208743
4,21+,172,0.162791


In [19]:
# ICU vs non‑ICU

icu_readmit = (
    base.groupby("icu_admitted")[target_col]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"mean": "readmission_rate"})
)

icu_readmit

,icu_admitted,count,readmission_rate
0,0,6569,0.211904
1,1,1931,0.218022


In [20]:
# Discharge disposition

disp_readmit = (
    base.groupby("discharge_disposition")[target_col]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"mean": "readmission_rate"})
    .sort_values("readmission_rate", ascending=False)
)

disp_readmit.head(10)

,discharge_disposition,count,readmission_rate
1,Expired,179,0.229050
0,AMA,245,0.220408
6,Skilled Nursing Facility,1141,0.216477
2,Home,3581,0.213907
5,Rehab Facility,660,0.212121
3,Home with Home Health,2137,0.211511
4,Long-term Care,557,0.202873


In [21]:
# Hospital‑level patterns
hospital_readmit = (
    base.groupby("hospital")[target_col]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"mean": "readmission_rate"})
    .sort_values("readmission_rate", ascending=False)
)

hospital_readmit

,hospital,count,readmission_rate
7,MHN South Shore,1009,0.237859
6,MHN Roxbury Community,1029,0.223518
5,MHN Quincy,1063,0.221072
2,MHN Dorchester,1070,0.214019
1,MHN Cambridge,1068,0.212547
4,MHN Jamaica Plain,1039,0.207892
0,MHN Boston General,1139,0.204565
3,MHN Fenway,1083,0.187442


In [22]:
age_readmit.to_csv("../outputs/tables/readmission_by_age_band.csv", index=False)
los_readmit.to_csv("../outputs/tables/readmission_by_los_band.csv", index=False)
icu_readmit.to_csv("../outputs/tables/readmission_by_icu.csv", index=False)
disp_readmit.to_csv("../outputs/tables/readmission_by_disposition.csv", index=False)
hospital_readmit.to_csv("../outputs/tables/readmission_by_hospital.csv", index=False)

In [25]:
# 1. Create flags on the merged table
base["readmitted_30d"] = (
    base["readmission_date"].notna()
    & (base["days_to_readmission"].fillna(np.inf) <= 30)
)

base["planned_readmission"] = base["planned_readmission"].fillna(0).astype(int)

base["unplanned_readmission_30d"] = (
    base["readmitted_30d"] & (base["planned_readmission"] == 0)
)

# optional avoidable flag, if present
if "avoided_if_discharged_better" in base.columns:
    base["avoidable_readmission_30d"] = (
        base["readmitted_30d"]
        & (base["avoided_if_discharged_better"].fillna(0).astype(int) == 1)
    )
else:
    base["avoidable_readmission_30d"] = False

In [26]:
summary_flags = pd.DataFrame({
    "overall_readmission_rate": [base["readmitted_30d"].mean()],
    "unplanned_readmission_rate": [base["unplanned_readmission_30d"].mean()],
    "planned_readmission_rate": [
        (
            (base["readmitted_30d"] == 1)
            & (base["planned_readmission"].fillna(0) == 1)
        ).mean()
    ],
})

summary_flags

,overall_readmission_rate,unplanned_readmission_rate,planned_readmission_rate
0,0.213294,0.186824,0.026471
